In [ ]:
from emu_renewal.outputs import get_output_dir, get_combined_df, get_df_from_3darray
import arviz as az
import pandas as pd
import numpy as np
import seaborn as sns

In [ ]:
country = "Spain"
analysis_times = {
    "no_mob": "20250116_2110",
    "google_nonresi_linear": "20250116_2113",
    "google_nonresi_square": "20250116_2117",
}

In [ ]:
idatas = {k: az.from_netcdf(get_output_dir(country, k, v) / "idata.nc") for k, v in analysis_times.items()}

In [ ]:
# Get the variable process and dispersion values, with samples as first axis (for use as index to dataframe later)
mob_proc_vals = np.swapaxes(idatas["google_nonresi_linear"].posterior["proc"].to_numpy(), 0, 1)
mob_disp_vals = np.swapaxes(idatas["google_nonresi_linear"].posterior["dispersion_proc"].to_numpy(), 0, 1)
no_mob_proc_vals = np.swapaxes(idatas["no_mob"].posterior["proc"].to_numpy(), 0, 1)
no_mob_disp_vals = np.swapaxes(idatas["no_mob"].posterior["dispersion_proc"].to_numpy(), 0, 1)

# Get dataframes from arrays
mob_proc_df = get_df_from_3darray(mob_proc_vals, [0, 2, 1])
no_mob_proc_df = get_df_from_3darray(no_mob_proc_vals, [0, 2, 1])

# Combine the mobility and no mobility dataframes
proc_comparison_df = get_combined_df(mob_proc_df, no_mob_proc_df, "mob", "no_mob")

# Combine the dispersion values into a dataframe
disp_vals_dict = {
    "mob": mob_disp_vals.flatten(),
    "no_mob": no_mob_disp_vals.flatten(),
}
disp_comparison_df = pd.DataFrame(disp_vals_dict)

In [ ]:
sns.kdeplot(disp_comparison_df, fill=True)

In [ ]:
sns.kdeplot(proc_comparison_df, fill=True)